# 🩸 Glucast — Modelo LSTM de Predicción de Glucemia
## Trabajo Final de Maestría en Inteligencia Artificial — UNIR
**Autor:** Pablo Daniel Figueroa  
**Institución:** Universidad Nacional de Tucumán (UNT) / UNIR  

Este notebook entrena un modelo LSTM para predecir la glucemia futura (30-60 minutos)
usando datos multivariables:
- 🩸 Glucosa CGM (Medtronic Guardian 4 via CareLink)
- 💉 Bolos de insulina con timestamp exacto
- ⌚ Estrés, sueño, FC y actividad (Garmin Instinct)


## 1. Instalación de dependencias

In [ ]:
!pip install pandas numpy matplotlib scikit-learn tensorflow keras seaborn -q
print('✅ Dependencias instaladas')

## 2. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('✅ Importaciones completadas')

## 3. Subir los archivos CSV

Subí los siguientes archivos desde tu PC:
- `Pablo Figueroa DD-MM-YY.csv` (exportado de CareLink)
- `garmin_datos_consolidados.csv` (generado por procesar_garmin.py)

In [ ]:
from google.colab import files
print('Subí el CSV de CareLink y el CSV de Garmin')
uploaded = files.upload()

## 4. Carga y limpieza del CSV de CareLink (CGM + Bolos)

In [ ]:
import os

# Detectar el archivo CSV de CareLink automáticamente
carelink_file = [f for f in uploaded.keys() if 'Figueroa' in f or 'Pablo' in f]
garmin_file = [f for f in uploaded.keys() if 'garmin' in f.lower()]

if not carelink_file:
    raise ValueError('No se encontró el CSV de CareLink. Verificá el nombre del archivo.')
if not garmin_file:
    raise ValueError('No se encontró el CSV de Garmin.')

carelink_file = carelink_file[0]
garmin_file = garmin_file[0]

print(f'CareLink: {carelink_file}')
print(f'Garmin: {garmin_file}')

In [ ]:
def parsear_carelink(filename):
    """
    Parsea el CSV de CareLink extrayendo:
    - Lecturas de glucosa cada 5 minutos
    - Bolos de insulina con timestamp exacto
    """
    import csv
    from datetime import datetime, timezone

    glucosa_rows = []
    bolo_rows = []
    headers = None

    with open(filename, encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        rows = list(reader)

    date_col = time_col = sgv_col = bolus_col = None

    for row in rows:
        if not row:
            continue
        if 'Date' in row and 'Time' in row and 'Sensor Glucose (mg/dL)' in row:
            headers = row
            date_col = headers.index('Date')
            time_col = headers.index('Time')
            sgv_col = headers.index('Sensor Glucose (mg/dL)')
            bolus_col = headers.index('Bolus Volume Delivered (U)') if 'Bolus Volume Delivered (U)' in headers else None
            continue

        if headers is None or len(row) <= max(filter(None, [date_col, time_col, sgv_col])):
            continue

        date_str = row[date_col].strip()
        time_str = row[time_col].strip()
        if not date_str or not time_str:
            continue

        try:
            dt = datetime.strptime(f'{date_str} {time_str}', '%Y/%m/%d %H:%M:%S')
        except ValueError:
            continue

        # Glucosa
        sgv_str = row[sgv_col].strip() if sgv_col < len(row) else ''
        if sgv_str.isdigit():
            glucosa_rows.append({'timestamp': dt, 'glucosa': int(sgv_str)})

        # Bolo
        if bolus_col and bolus_col < len(row):
            bolus_str = row[bolus_col].strip()
            if bolus_str:
                try:
                    u = float(bolus_str)
                    if u > 0:
                        bolo_rows.append({'timestamp': dt, 'bolo_u': u})
                except ValueError:
                    pass

    df_gluc = pd.DataFrame(glucosa_rows).set_index('timestamp').sort_index()
    df_bolos = pd.DataFrame(bolo_rows).set_index('timestamp').sort_index()

    print(f'Lecturas de glucosa: {len(df_gluc)}')
    print(f'Bolos de insulina: {len(df_bolos)}')
    print(f'Período: {df_gluc.index.min()} → {df_gluc.index.max()}')

    return df_gluc, df_bolos

df_gluc, df_bolos = parsear_carelink(carelink_file)
df_gluc.head()

## 5. Carga del CSV de Garmin

In [ ]:
df_garmin = pd.read_csv(garmin_file)
df_garmin['fecha'] = pd.to_datetime(df_garmin['fecha'])
df_garmin = df_garmin.set_index('fecha').sort_index()

# Seleccionar variables relevantes para el modelo
cols_garmin = ['estres_promedio', 'fc_reposo', 'sueno_total_horas', 'pasos_totales']
cols_disponibles = [c for c in cols_garmin if c in df_garmin.columns]
df_garmin = df_garmin[cols_disponibles]

print(f'Días de datos Garmin: {len(df_garmin)}')
print(f'Variables: {list(df_garmin.columns)}')
df_garmin.head()

## 6. Construcción del dataset unificado en resolución de 5 minutos

In [ ]:
# Resamplear glucosa a 5 minutos
df_base = df_gluc.resample('5min').mean().interpolate(method='linear', limit=3)

# Agregar bolos de insulina — sumar bolos en cada ventana de 5 min
if len(df_bolos) > 0:
    df_bolos_5m = df_bolos.resample('5min').sum()
    df_base = df_base.join(df_bolos_5m, how='left')
    df_base['bolo_u'] = df_base['bolo_u'].fillna(0)
else:
    df_base['bolo_u'] = 0

# --- Feature engineering ---

# 1. Insulina activa acumulada (IOB estimado — DIA = 2h = 24 períodos de 5 min)
# Modelo simple: la insulina decae linealmente en 2 horas
DIA_PERIODOS = 24  # 2 horas / 5 min
pesos_iob = np.array([max(0, 1 - i/DIA_PERIODOS) for i in range(DIA_PERIODOS)])
df_base['iob'] = df_base['bolo_u'].rolling(DIA_PERIODOS, min_periods=1).apply(
    lambda x: np.sum(x[::-1] * pesos_iob[:len(x)]), raw=True
)

# 2. Tendencia de glucosa (diferencia con lectura anterior)
df_base['tendencia'] = df_base['glucosa'].diff()

# 3. Hora del día (representación cíclica)
hora = df_base.index.hour + df_base.index.minute / 60
df_base['hora_sin'] = np.sin(2 * np.pi * hora / 24)
df_base['hora_cos'] = np.cos(2 * np.pi * hora / 24)

# 4. Día de la semana (cíclico)
dow = df_base.index.dayofweek
df_base['dow_sin'] = np.sin(2 * np.pi * dow / 7)
df_base['dow_cos'] = np.cos(2 * np.pi * dow / 7)

# 5. Fenómeno del alba (flag entre 5-8 AM)
df_base['alba'] = ((df_base.index.hour >= 5) & (df_base.index.hour < 8)).astype(int)

# 6. Unir datos de Garmin (granularidad diaria → expandir a 5 min)
df_base['fecha'] = df_base.index.normalize()
df_garmin_reset = df_garmin.reset_index()
df_garmin_reset.columns = ['fecha'] + list(df_garmin.columns)
df_base = df_base.reset_index().merge(df_garmin_reset, on='fecha', how='left').set_index('timestamp')
df_base = df_base.drop(columns=['fecha'])

# Rellenar datos Garmin faltantes con la media
for col in cols_disponibles:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(df_base[col].mean())

# Eliminar filas con NaN en glucosa
df_base = df_base.dropna(subset=['glucosa'])
df_base['tendencia'] = df_base['tendencia'].fillna(0)

print(f'Dataset unificado: {len(df_base)} filas x {len(df_base.columns)} columnas')
print(f'Columnas: {list(df_base.columns)}')
df_base.describe()

## 7. Visualización exploratoria del dataset

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Glucosa
axes[0].plot(df_base.index[-2016:], df_base['glucosa'][-2016:], color='#1a73e8', linewidth=0.8)
axes[0].axhline(70, color='red', linestyle='--', alpha=0.5, label='Hipoglucemia')
axes[0].axhline(180, color='orange', linestyle='--', alpha=0.5, label='Hiperglucemia')
axes[0].set_ylabel('Glucosa (mg/dL)')
axes[0].set_title('Glucemia últimos 7 días')
axes[0].legend()

# IOB
axes[1].plot(df_base.index[-2016:], df_base['iob'][-2016:], color='#e53935', linewidth=0.8)
axes[1].set_ylabel('IOB estimado (U)')
axes[1].set_title('Insulina Activa (IOB)')

# Estrés Garmin
if 'estres_promedio' in df_base.columns:
    axes[2].plot(df_base.index[-2016:], df_base['estres_promedio'][-2016:], color='#43a047', linewidth=0.8)
    axes[2].set_ylabel('Estrés Garmin')
    axes[2].set_title('Nivel de estrés diario')

plt.tight_layout()
plt.savefig('exploracion_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado')

## 8. Preparación de secuencias para LSTM

El modelo predice la glucosa a **30 minutos** (6 períodos de 5 min) usando una ventana de **2 horas** de historia (24 períodos).

In [ ]:
# Parámetros del modelo
VENTANA = 24        # 2 horas de historia (24 × 5 min)
HORIZONTE = 6       # Predicción a 30 minutos (6 × 5 min)
TEST_RATIO = 0.2    # 20% para test

# Seleccionar features
FEATURES = ['glucosa', 'bolo_u', 'iob', 'tendencia',
            'hora_sin', 'hora_cos', 'dow_sin', 'dow_cos', 'alba']

# Agregar features de Garmin disponibles
for col in cols_disponibles:
    if col in df_base.columns:
        FEATURES.append(col)

TARGET = 'glucosa'

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Ventana: {VENTANA} períodos ({VENTANA*5} min)')
print(f'Horizonte de predicción: {HORIZONTE} períodos ({HORIZONTE*5} min)')

# Escalar datos
df_model = df_base[FEATURES].dropna()
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df_model)

# Índice de la columna target
target_idx = FEATURES.index(TARGET)

# Crear secuencias X (entrada) e y (salida)
X, y = [], []
for i in range(VENTANA, len(data_scaled) - HORIZONTE):
    X.append(data_scaled[i-VENTANA:i])
    y.append(data_scaled[i+HORIZONTE-1, target_idx])

X = np.array(X)
y = np.array(y)

# Split train/test (sin shuffle — datos temporales)
split = int(len(X) * (1 - TEST_RATIO))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'\nTotal secuencias: {len(X)}')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Shape X: {X.shape} | Shape y: {y.shape}')

## 9. Construcción del modelo LSTM

In [ ]:
def construir_modelo(input_shape):
    model = Sequential([
        # Primera capa LSTM
        LSTM(64, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.2),

        # Segunda capa LSTM
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),

        # Capas densas
        Dense(16, activation='relu'),
        Dense(1)  # Predicción de glucosa
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

modelo = construir_modelo((VENTANA, len(FEATURES)))
modelo.summary()

## 10. Entrenamiento

In [ ]:
callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=7, factor=0.5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('glucast_modelo_lstm.h5', save_best_only=True, verbose=0)
]

historia = modelo.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Entrenamiento completado')

## 11. Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(historia.history['loss'], label='Train Loss')
axes[0].plot(historia.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(historia.history['mae'], label='Train MAE')
axes[1].plot(historia.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE (Error Absoluto Medio)')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.savefig('curvas_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Evaluación del modelo

In [ ]:
# Predicciones en el conjunto de test
y_pred_scaled = modelo.predict(X_test, verbose=0)

# Invertir escalado para obtener valores en mg/dL
def invertir_escalado(values_scaled, scaler, feature_idx, n_features):
    dummy = np.zeros((len(values_scaled), n_features))
    dummy[:, feature_idx] = values_scaled.flatten()
    return scaler.inverse_transform(dummy)[:, feature_idx]

y_pred = invertir_escalado(y_pred_scaled, scaler, target_idx, len(FEATURES))
y_real = invertir_escalado(y_test, scaler, target_idx, len(FEATURES))

# Métricas
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
mae = mean_absolute_error(y_real, y_pred)

# Clarke Error Grid (simplificado)
# Zona A: error < 20% o ambos < 70 mg/dL
zona_a = np.mean(
    (np.abs(y_real - y_pred) / np.maximum(y_real, 1) < 0.20) |
    ((y_real < 70) & (y_pred < 70))
) * 100

# TIR (Time In Range 70-180)
tir_real = np.mean((y_real >= 70) & (y_real <= 180)) * 100
tir_pred = np.mean((y_pred >= 70) & (y_pred <= 180)) * 100

print('=' * 50)
print('📊 MÉTRICAS DEL MODELO GLUCAST LSTM')
print('=' * 50)
print(f'RMSE:              {rmse:.2f} mg/dL')
print(f'MAE:               {mae:.2f} mg/dL')
print(f'Clarke Zona A:     {zona_a:.1f}%')
print(f'TIR real:          {tir_real:.1f}%')
print(f'TIR predicho:      {tir_pred:.1f}%')
print('=' * 50)

# Referencia clínica: RMSE < 20 mg/dL es clínicamente aceptable
if rmse < 20:
    print('✅ RMSE dentro del rango clínicamente aceptable (< 20 mg/dL)')
else:
    print('⚠️ RMSE por encima del umbral clínico — considerar más datos o ajustar arquitectura')

## 13. Visualización de predicciones vs valores reales

In [ ]:
# Mostrar las últimas 72 horas del conjunto de test
n_mostrar = min(864, len(y_real))  # 72h × 12 períodos/h

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Predicción vs real
axes[0].plot(y_real[-n_mostrar:], label='Glucosa real', color='#1a73e8', linewidth=1)
axes[0].plot(y_pred[-n_mostrar:], label='Predicción LSTM (+30 min)', color='#e53935',
             linewidth=1, linestyle='--')
axes[0].axhline(70, color='red', alpha=0.3, linestyle=':')
axes[0].axhline(180, color='orange', alpha=0.3, linestyle=':')
axes[0].fill_between(range(n_mostrar), 70, 180, alpha=0.05, color='green', label='Rango objetivo')
axes[0].set_ylabel('Glucosa (mg/dL)')
axes[0].set_title(f'Predicción LSTM a 30 min — RMSE: {rmse:.1f} mg/dL | MAE: {mae:.1f} mg/dL')
axes[0].legend()

# Error de predicción
error = y_pred[-n_mostrar:] - y_real[-n_mostrar:]
axes[1].bar(range(len(error)), error, color=np.where(np.abs(error) > 20, '#e53935', '#43a047'),
            alpha=0.6, width=1)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(20, color='orange', linestyle='--', alpha=0.5, label='+20 mg/dL')
axes[1].axhline(-20, color='orange', linestyle='--', alpha=0.5, label='-20 mg/dL')
axes[1].set_ylabel('Error (mg/dL)')
axes[1].set_title('Error de predicción (verde < 20 mg/dL)')
axes[1].legend()

plt.tight_layout()
plt.savefig('predicciones_lstm.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Distribución de errores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de errores
errores = y_pred - y_real
axes[0].hist(errores, bins=50, color='#1a73e8', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].axvline(mae, color='orange', linestyle='--', label=f'MAE = {mae:.1f}')
axes[0].axvline(-mae, color='orange', linestyle='--')
axes[0].set_xlabel('Error (mg/dL)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de errores')
axes[0].legend()

# Scatter real vs predicho
axes[1].scatter(y_real, y_pred, alpha=0.1, s=2, color='#1a73e8')
min_val, max_val = min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', label='Predicción perfecta')
axes[1].plot([min_val, max_val], [min_val+20, max_val+20], 'orange', alpha=0.5, linestyle=':')
axes[1].plot([min_val, max_val], [min_val-20, max_val-20], 'orange', alpha=0.5, linestyle=':')
axes[1].set_xlabel('Glucosa real (mg/dL)')
axes[1].set_ylabel('Glucosa predicha (mg/dL)')
axes[1].set_title('Real vs Predicho')
axes[1].legend()

plt.tight_layout()
plt.savefig('distribucion_errores.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Guardar modelo y scaler

In [ ]:
import pickle
import json

# Guardar scaler
with open('glucast_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Guardar metadatos del modelo
metadata = {
    'features': FEATURES,
    'target': TARGET,
    'ventana': VENTANA,
    'horizonte': HORIZONTE,
    'rmse': float(rmse),
    'mae': float(mae),
    'zona_a': float(zona_a),
    'n_train': len(X_train),
    'n_test': len(X_test),
}
with open('glucast_modelo_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Archivos guardados:')
print('  - glucast_modelo_lstm.h5')
print('  - glucast_scaler.pkl')
print('  - glucast_modelo_metadata.json')
print()
print(json.dumps(metadata, indent=2))

## 16. Descargar archivos del modelo

In [ ]:
from google.colab import files

files.download('glucast_modelo_lstm.h5')
files.download('glucast_scaler.pkl')
files.download('glucast_modelo_metadata.json')
files.download('predicciones_lstm.png')
files.download('curvas_aprendizaje.png')
files.download('distribucion_errores.png')

print('✅ Archivos descargados')

## 17. Resumen final

Este notebook entrenó un modelo LSTM para predecir glucemia a **30 minutos** usando:
- Datos CGM de CareLink (Medtronic MiniMed 780G + Guardian 4)
- Bolos de insulina con timestamp exacto
- IOB estimado (insulina activa)
- Tendencia de glucosa
- Variables temporales cíclicas (hora del día, día de la semana)
- Flag de fenómeno del alba (5-8 AM)
- Estrés, FC en reposo, sueño y pasos (Garmin Instinct)

**Próximos pasos:**
1. Integrar el modelo en Glucast para predicciones en tiempo real
2. Agregar más datos para mejorar la precisión
3. Explorar arquitecturas Transformer para comparar resultados
4. Implementar predicción a 60 minutos (horizonte = 12 períodos)
